# Практика 2. Работа с конвейерами Hugging Face

Выполнение заданий второй части практикума.

In [23]:
# pip install transformers datasets torch pandas tabulate --quiet


import torch
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification, AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM
from tabulate import tabulate
import gc
import time
import numpy as np

In [30]:
# ============================================================
# 5. Распознавание именованных сущностей (NER)
# Задание 1. 7 моделей, 10 предложений (альтернативный вариант)
# ============================================================

print("="*70)
print("ЗАДАНИЕ 1. NER: 7 моделей, 10 предложений (без pandas)")
print("="*70)

# ---- 1.1 Подготовка тестовых предложений и экспертной разметки ----
test_sentences_ner = [
    "Президент России Владимир Путин 24 февраля 2022 года объявил о начале спецоперации.",
    "Компания Apple выпустила iPhone 15 в сентябре 2023 года в США.",
    "Лев Толстой написал роман «Анна Каренина» в XIX веке.",
    "Самолёт Boeing 737 авиакомпании S7 совершил экстренную посадку в Новосибирске 12 июля.",
    "Глава МИД РФ Сергей Лавров встретился с госсекретарём США Энтони Блинкеном в Нью-Йорке.",
    "Олимпийские игры 2024 года пройдут в Париже, Франция.",
    "Байкал — самое глубокое озеро, находится в Иркутской области.",
    "В 2023 году Нобелевскую премию по литературе получил Юн Фоссе.",
    "ЦБ РФ повысил ключевую ставку до 16% годовых 15 декабря.",
    "Космический корабль «Союз МС-24» стартовал с Байконура 15 сентября."
]

# Экспертная разметка (ground truth) для каждого предложения
gold_ner = [
    [("Владимир Путин", "PER"), ("24 февраля 2022 года", "DATE")],
    [("Apple", "ORG"), ("iPhone 15", "PROD"), ("сентября 2023 года", "DATE"), ("США", "LOC")],
    [("Лев Толстой", "PER"), ("«Анна Каренина»", "WORK"), ("XIX веке", "DATE")],
    [("Boeing 737", "PROD"), ("S7", "ORG"), ("Новосибирске", "LOC"), ("12 июля", "DATE")],
    [("Сергей Лавров", "PER"), ("МИД РФ", "ORG"), ("Энтони Блинкен", "PER"), ("США", "ORG"), ("Нью-Йорке", "LOC")],
    [("Олимпийские игры 2024 года", "EVENT"), ("Париже", "LOC"), ("Франция", "LOC")],
    [("Байкал", "LOC"), ("Иркутской области", "LOC")],
    [("2023 году", "DATE"), ("Нобелевскую премию по литературе", "AWARD"), ("Юн Фоссе", "PER")],
    [("ЦБ РФ", "ORG"), ("16%", "PERCENT"), ("15 декабря", "DATE")],
    [("«Союз МС-24»", "PROD"), ("Байконура", "LOC"), ("15 сентября", "DATE")]
]

# ---- 1.2 Список моделей NER (7 новых моделей) ----
ner_models = [
    "julian-schelb/roberta-ner-multilingual",
    "ivlcic/xlmr-ner-slavic",
    "Babelscape/wikineural-multilingual-ner",
    "surdan/LaBSE_ner_nerel",
    "dmargutierrez/distilbert-base-multilingual-cased-mapa_coarse-ner",
    "dslim/bert-base-NER"
]

def normalize_entity(text):
    """Нормализация текста сущности (нижний регистр, убираем лишние пробелы)."""
    return text.strip().lower().replace("  ", " ")

def evaluate_ner(pipeline, sentences, gold):
    """Возвращает точность (accuracy) и детализированные результаты."""
    correct = 0
    total = 0
    details = []
    for i, sent in enumerate(sentences):
        preds = pipeline(sent, aggregation_strategy="simple")
        pred_set = set()
        for p in preds:
            word = p['word']
            etype = p.get('entity_group', p.get('entity', 'MISC'))
            if etype.startswith(('B-', 'I-')):
                etype = etype[2:]
            pred_set.add((normalize_entity(word), etype))
        gold_set = set((normalize_entity(t), typ) for t, typ in gold[i])
        matches = pred_set & gold_set
        correct += len(matches)
        total += len(gold_set)
        details.append({
            "Предложение": sent[:40] + "...",
            "Найдено": ", ".join([f"{t} ({typ})" for t, typ in pred_set])[:70],
            "Ожидалось": ", ".join([f"{t} ({typ})" for t, typ in gold_set])[:70],
            "Совпадений": f"{len(matches)}/{len(gold_set)}"
        })
    accuracy = (correct / total * 100) if total > 0 else 0
    return accuracy, details

results_ner = []   # список для итоговой таблицы [модель, точность]

print("\nЗагрузка моделей NER и оценка...")
for model_name in ner_models:
    print(f"  {model_name}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForTokenClassification.from_pretrained(model_name)
        ner_pipe = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple",
            device=0 if torch.cuda.is_available() else -1
        )
        acc, _ = evaluate_ner(ner_pipe, test_sentences_ner, gold_ner)
        results_ner.append([model_name, f"{acc:.2f}%"])
        # Очистка памяти
        del model, tokenizer, ner_pipe
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"    Ошибка с моделью {model_name}: {e}")
        results_ner.append([model_name, "ошибка"])

# ---- Вывод итоговой таблицы через tabulate ----
print("\nИТОГОВАЯ ТАБЛИЦА (NER):")
print(tabulate(results_ner, headers=["Модель", "Точность"], tablefmt="grid"))

# При желании можно также вывести детали для лучшей модели
if results_ner:
    # Найдём модель с максимальной точностью (если не ошибка)
    valid = [r for r in results_ner if r[1] != "ошибка"]
    if valid:
        best_model = max(valid, key=lambda x: float(x[1].strip('%')))
        print(f"\nЛучшая модель: {best_model[0]} ({best_model[1]})")

ЗАДАНИЕ 1. NER: 7 моделей, 10 предложений (без pandas)

Загрузка моделей NER и оценка...
  julian-schelb/roberta-ner-multilingual...


config.json: 0.00B [00:00, ?B/s]

d:\julia\Bachelor\4_course\2sem\Моделирование процессов в нотации BPMN\labs_bpmn\huggingface_labs\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jylij\.cache\huggingface\hub\models--julian-schelb--roberta-ner-multilingual. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/499 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

  dslim/bert-base-NER...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



ИТОГОВАЯ ТАБЛИЦА (NER):
+------------------------------------------------------------------+------------+
| Модель                                                           | Точность   |
+==================================================================+============+
| julian-schelb/roberta-ner-multilingual                           | 18.75%     |
+------------------------------------------------------------------+------------+
| ivlcic/xlmr-ner-slavic                                           | 18.75%     |
+------------------------------------------------------------------+------------+
| Babelscape/wikineural-multilingual-ner                           | 46.88%     |
+------------------------------------------------------------------+------------+
| surdan/LaBSE_ner_nerel                                           | 15.62%     |
+------------------------------------------------------------------+------------+
| dmargutierrez/distilbert-base-multilingual-cased-mapa_coarse-ner | 12.5

In [33]:
# ============================================================
# 6. Ответы на вопросы из контекста (QA)
# Задание 2. Подберите 5 моделей QA и протестируйте на 3 контекстах
# ============================================================

print("\n" + "="*70)
print("ЗАДАНИЕ 2. QA: 5 моделей, 3 контекста с вопросами")
print("="*70)

# ---- 2.1 Подготовка контекстов и вопросов с правильными ответами ----
qa_data = [
    {
        "context": "Байкал — самое глубокое озеро на Земле. Оно расположено в Восточной Сибири. Возраст озера оценивается примерно в 25 миллионов лет.",
        "qa": [
            ("Где находится озеро Байкал?", "Восточной Сибири"),
            ("Какое озеро самое глубокое?", "Байкал"),
            ("Сколько лет озеру Байкал?", "25 миллионов лет")
        ]
    },
    {
        "context": "Александр Сергеевич Пушкин родился в Москве 6 июня 1799 года. Он считается основоположником современного русского литературного языка.",
        "qa": [
            ("Когда родился Пушкин?", "6 июня 1799 года"),
            ("В каком городе родился Пушкин?", "Москве"),
            ("Кем считается Пушкин?", "основоположником современного русского литературного языка")
        ]
    },
    {
        "context": "Солнце — звезда, которая находится в центре Солнечной системы. Земля вращается вокруг Солнца примерно за 365 дней.",
        "qa": [
            ("Что находится в центре Солнечной системы?", "Солнце"),
            ("За сколько дней Земля вращается вокруг Солнца?", "365 дней")
        ]
    }
]

# ---- 2.2 Список моделей QA (5 штук) ----
qa_models = [
    "SriusG/SiriusBetta",
    "nicoboss/Apollo2-9B-llamacppfixed",
    "QuantFactory/Apollo2-2B-GGUF",
    "QuantFactory/Apollo2-7B-GGUF",
    "mradermacher/next-12b-i1-GGUF"
]


def evaluate_qa(pipeline, qa_data):
    """Оценивает модель QA: для каждой пары вопрос-контекст сравнивает ответ с ожидаемым.
       Возвращает точность (exact match) и детали."""
    correct = 0
    total = 0
    details = []
    for item in qa_data:
        ctx = item["context"]
        for question, expected in item["qa"]:
            result = pipeline(question=question, context=ctx, top_k=1)
            answer = result['answer'].strip().lower()
            exp = expected.strip().lower()
            match = 1 if answer in exp or exp in answer else 0  # нестрогое сравнение
            correct += match
            total += 1
            details.append({
                "Контекст": ctx[:40] + "...",
                "Вопрос": question,
                "Ожидаемый ответ": expected,
                "Ответ модели": result['answer'],
                "Уверенность": f"{result['score']:.2%}",
                "Совпадение": match
            })
    accuracy = correct / total * 100
    return accuracy, details

results_qa = []
detailed_qa = {}

print("\nЗагрузка моделей QA и оценка...")
for model_name in qa_models:
    print(f"  {model_name}...")
    try:
        qa_pipe = pipeline("question-answering", model=model_name, device=0 if torch.cuda.is_available() else -1)
        acc, det = evaluate_qa(qa_pipe, qa_data)
        results_qa.append({"Модель": model_name, "Точность (%)": round(acc, 2)})
        detailed_qa[model_name] = det
        del qa_pipe
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f"    Ошибка: {e}")
        results_qa.append({"Модель": model_name, "Точность (%)": 0.0})

df_qa = pd.DataFrame(results_qa).sort_values("Точность (%)", ascending=False)
print("\nИТОГОВАЯ ТАБЛИЦА (QA):")
print(tabulate(df_qa, headers='keys', tablefmt='grid', showindex=False))


ЗАДАНИЕ 2. QA: 5 моделей, 3 контекста с вопросами

Загрузка моделей QA и оценка...
  SriusG/SiriusBetta...
    Ошибка: Unrecognized model in SriusG/SiriusBetta. Should have a `model_type` key in its config.json.
  nicoboss/Apollo2-9B-llamacppfixed...


config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

d:\julia\Bachelor\4_course\2sem\Моделирование процессов в нотации BPMN\labs_bpmn\huggingface_labs\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jylij\.cache\huggingface\hub\models--nicoboss--Apollo2-9B-llamacppfixed. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


    Ошибка: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"
  QuantFactory/Apollo2-2B-GGUF...
    Ошибка: Unrecognized model in QuantFactory/Apollo2-2B-GGUF. Should have a `model_type` key in its config.json.
  QuantFactory/Apollo2-7B-GGUF...
    Ошибка: Unrecognized model in QuantFactory/Apollo2-7B-GGUF. Should have a `model_type` key in its config.json.
  mradermacher

In [34]:
# ============================================================
# Задание 3. Влияние параметров QA (на примере одной модели)
# ============================================================

print("\n" + "="*70)
print("ЗАДАНИЕ 3. Эксперименты с параметрами QA (на модели deepset/xlm-roberta-base-squad2)")
print("="*70)

model_param = "deepset/xlm-roberta-base-squad2"
qa_pipe = pipeline("question-answering", model=model_param, device=0 if torch.cuda.is_available() else -1)

# Наборы параметров
param_sets = [
    {"name": "Базовые", "top_k": 1, "doc_stride": 128, "max_answer_len": 30, "max_seq_len": 384, "max_question_len": 64},
    {"name": "top_k=3", "top_k": 3, "doc_stride": 128, "max_answer_len": 30, "max_seq_len": 384, "max_question_len": 64},
    {"name": "doc_stride=64", "top_k": 1, "doc_stride": 64, "max_answer_len": 30, "max_seq_len": 384, "max_question_len": 64},
    {"name": "max_answer_len=10", "top_k": 1, "doc_stride": 128, "max_answer_len": 10, "max_seq_len": 384, "max_question_len": 64},
    {"name": "max_seq_len=256", "top_k": 1, "doc_stride": 128, "max_answer_len": 30, "max_seq_len": 256, "max_question_len": 64},
    {"name": "max_question_len=10", "top_k": 1, "doc_stride": 128, "max_answer_len": 30, "max_seq_len": 384, "max_question_len": 10},
]

results_params = []
for params in param_sets:
    print(f"\nТестирование: {params['name']}")
    correct = 0
    total = 0
    for item in qa_data:
        ctx = item["context"]
        for question, expected in item["qa"]:
            result = qa_pipe(
                question=question,
                context=ctx,
                top_k=params["top_k"],
                doc_stride=params["doc_stride"],
                max_answer_len=params["max_answer_len"],
                max_seq_len=params["max_seq_len"],
                max_question_len=params["max_question_len"]
            )
            # Если top_k>1, берём лучший ответ
            if isinstance(result, list):
                answer = result[0]['answer']
                score = result[0]['score']
            else:
                answer = result['answer']
                score = result['score']
            exp = expected.strip().lower()
            match = 1 if answer.strip().lower() in exp or exp in answer.strip().lower() else 0
            correct += match
            total += 1
    acc = correct / total * 100
    results_params.append({"Параметры": params["name"], "Точность (%)": round(acc, 2)})

df_params = pd.DataFrame(results_params)
print("\nВЛИЯНИЕ ПАРАМЕТРОВ НА ТОЧНОСТЬ:")
print(tabulate(df_params, headers='keys', tablefmt='grid', showindex=False))



ЗАДАНИЕ 3. Эксперименты с параметрами QA (на модели deepset/xlm-roberta-base-squad2)


config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [35]:
# ============================================================
# 7. Аннотирование текстов (суммаризация)
# Задания 4 и 5: разные модели, параметры, модификация кода с pipeline
# ============================================================

print("\n" + "="*70)
print("ЗАДАНИЯ 4 и 5. Суммаризация: 3 модели, эксперименты с параметрами")
print("="*70)

# ---- Новый текст для суммаризации (про Амазонку) ----
text_amazon = """
Амазонка — самая полноводная река в мире, расположенная в Южной Америке. Она протекает через территории Перу, Колумбии и Бразилии. Длина Амазонки составляет около 7100 километров, что делает её второй по длине после Нила, однако по объёму воды она превосходит все остальные реки. Бассейн Амазонки покрывает примерно 7 миллионов квадратных километров и включает в себя крупнейший на Земле тропический лес, который производит около 20% кислорода планеты. В реке обитает огромное количество видов рыб, включая знаменитую пиранью, а также розовые дельфины, анаконды и кайманы. Коренные народы живут вдоль Амазонки тысячи лет, сохраняя традиционный образ жизни. Однако в последние десятилетия экосистема региона страдает от вырубки лесов, незаконной добычи золота и строительства гидроэлектростанций. Международные организации призывают к защите Амазонии как важнейшего лёгкого планеты.
"""

# ---- 3 модели суммаризации (русскоязычные) ----
summ_models = [
    "cointegrated/rut5-base-absum",
    "IlyaGusev/mbart_ru_sum_gazeta",
    "cointegrated/rut5-base-multitask"
]

# Наборы параметров для суммаризации
summ_params = [
    {"name": "Базовые", "max_length": 150, "min_length": 50, "num_beams": 4, "length_penalty": 2.0, "no_repeat_ngram_size": 3, "early_stopping": True},
    {"name": "Короткие", "max_length": 80, "min_length": 20, "num_beams": 4, "length_penalty": 1.5, "no_repeat_ngram_size": 3, "early_stopping": True},
    {"name": "Длинные", "max_length": 250, "min_length": 100, "num_beams": 4, "length_penalty": 1.0, "no_repeat_ngram_size": 3, "early_stopping": True},
    {"name": "Креативные (sampling)", "max_length": 150, "min_length": 50, "do_sample": True, "temperature": 0.9, "top_p": 0.9, "num_beams": 1, "repetition_penalty": 2.0},
]

results_summ = []

print("\nЗагрузка моделей суммаризации...")
for model_name in summ_models:
    print(f"  {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    # Используем pipeline summarization
    summarizer = pipeline("summarization", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

    for params in summ_params:
        print(f"    Параметры: {params['name']}")
        try:
            # Убираем параметры, несовместимые с pipeline
            pipe_params = {k: v for k, v in params.items() if k != "name" and k not in ["do_sample", "temperature", "top_p", "repetition_penalty"]}
            if params.get("do_sample"):
                pipe_params["do_sample"] = True
                pipe_params["temperature"] = params.get("temperature", 0.9)
                pipe_params["top_p"] = params.get("top_p", 0.9)
                pipe_params["repetition_penalty"] = params.get("repetition_penalty", 2.0)
            else:
                pipe_params["do_sample"] = False

            summary = summarizer(text_amazon, **pipe_params)[0]['summary_text']

            # Оценка длины и качества
            tokens_in = len(tokenizer.encode(text_amazon))
            tokens_out = len(tokenizer.encode(summary))
            compression = tokens_out / tokens_in * 100

            # Простая эвристика качества: наличие ключевых слов
            keywords = ["амазонк", "река", "лес", "вода", "южн", "бразил"]
            keyword_score = sum(1 for kw in keywords if kw in summary.lower()) / len(keywords) * 100

            results_summ.append({
                "Модель": model_name.split("/")[-1],
                "Параметры": params["name"],
                "Длина (токены)": tokens_out,
                "Сжатие (%)": round(compression, 1),
                "Качество (ключ.слова %)": round(keyword_score, 1),
                "Текст суммаризации": summary[:150] + "..."
            })
        except Exception as e:
            print(f"      Ошибка: {e}")

    del model, tokenizer, summarizer
    gc.collect()
    torch.cuda.empty_cache()

df_summ = pd.DataFrame(results_summ)
print("\nРЕЗУЛЬТАТЫ СУММАРИЗАЦИИ:")
print(tabulate(df_summ, headers='keys', tablefmt='grid', showindex=False))

# Выводы по параметрам
print("\nВЫВОДЫ ПО ПАРАМЕТРАМ СУММАРИЗАЦИИ:")
print("""
- num_beams=4 даёт более связные тексты, но медленнее.
- length_penalty > 1 (например, 2.0) стимулирует краткость, иногда обрывает предложения.
- no_repeat_ngram_size=3 помогает избежать зацикливаний.
- do_sample с temperature=0.9 даёт разнообразные, но иногда менее фактологичные варианты.
- Лучшее качество (по ключевым словам) достигается при балансе длины (сжатие 15-30%) и использовании beam search.
""")


ЗАДАНИЯ 4 и 5. Суммаризация: 3 модели, эксперименты с параметрами

Загрузка моделей суммаризации...
  cointegrated/rut5-base-absum...


config.json:   0%|          | 0.00/753 [00:00<?, ?B/s]

d:\julia\Bachelor\4_course\2sem\Моделирование процессов в нотации BPMN\labs_bpmn\huggingface_labs\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jylij\.cache\huggingface\hub\models--cointegrated--rut5-base-absum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/828k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/977M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [40]:
# ============================================================
# 8. Перевод текста
# Задание 6. Модель для перевода русский ↔ бурятский
# ============================================================

print("\n" + "="*70)
print("ЗАДАНИЕ 6. Перевод русский ↔ бурятский")
print("="*70)

# Модель из примера (доступна на HF)
model_name_bxr = "SaranaAbidueva/nllb-200-bxr-ru"
print(f"Загружаем модель {model_name_bxr}...")
tokenizer_bxr = AutoTokenizer.from_pretrained(model_name_bxr)
model_bxr = AutoModelForSeq2SeqLM.from_pretrained(model_name_bxr)

def translate(text, src_lang="rus_Cyrl", tgt_lang="bxr_Cyrl"):
    """Перевод с помощью NLLB. Для направления bxr->ru нужно поменять языки."""
    # В модели могут быть заданы языковые коды по умолчанию, но для NLLB можно указать
    # через токенизатор: tokenizer.src_lang = src_lang и добавить токен языка в generate.
    # Упростим: предполагаем, что модель обучена на обоих направлениях и определяет сама.
    inputs = tokenizer_bxr.encode(text, return_tensors="pt")
    outputs = model_bxr.generate(inputs)
    return tokenizer_bxr.decode(outputs[0], skip_special_tokens=True)

# Примеры перевода в обе стороны
examples = [
    ("Доброе утро!", "rus -> bxr"),
    ("Сайн байна, ямар боломж?", "bxr -> rus"),
    ("Как пройти в библиотеку?", "rus -> bxr"),
    ("Библиотекэ хүрэхэ яахаб?", "bxr -> rus"),
    ("Спасибо за помощь!", "rus -> bxr"),
]

print("\nПРИМЕРЫ ПЕРЕВОДА:")
for text, direction in examples:
    translated = translate(text)
    print(f"{direction}: '{text}' -> '{translated}'")

# Очистка
del model_bxr, tokenizer_bxr
gc.collect()
torch.cuda.empty_cache()

print("\n" + "="*70)
print("ВСЕ ЗАДАНИЯ ВЫПОЛНЕНЫ.")
print("="*70)


ЗАДАНИЕ 6. Перевод русский ↔ бурятский
Загружаем модель SaranaAbidueva/nllb-200-bxr-ru...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



ПРИМЕРЫ ПЕРЕВОДА:
rus -> bxr: 'Доброе утро!' -> 'Үглөөнэй мэндэ!'
bxr -> rus: 'Сайн байна, ямар боломж?' -> 'Сайн байна, ямар hайхан байдал бии болохоб?'
rus -> bxr: 'Как пройти в библиотеку?' -> 'Как добраться до библиотеки?'
bxr -> rus: 'Библиотекэ хүрэхэ яахаб?' -> 'Теэд, тэрэ номой байшань яажа хүртэхэб?'
rus -> bxr: 'Спасибо за помощь!' -> 'Амарhанай түлөө баярые хүргэнэб!'

ВСЕ ЗАДАНИЯ ВЫПОЛНЕНЫ.
